# IBMGO · Telekom regresyon test seçimi
Bir kampanya değiştiğinde hangi testleri tekrar çalıştıralım?

**Ana sunum** 50–100–250–500–1000 test ve 83–97–109 veri tohumlarının dayanıklılık arşividir. Notebook açılışında yeni optimizasyon başlamaz. Hücreleri sırayla çalıştırınca kayıtlı sonuçlar açılır.

`IBMGO_Ek_Olcek_Deneyi.ipynb` eski 500–1000–5000 ve A/B deneyini ayrı tutar.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'kod').is_dir():
    raise RuntimeError('Notebook’u IBMGO proje klasöründe açın.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from kod.deneyler.sunum_deneyi import (
    Settings, PresentationExperiment, summarize, result_figures,
    convergence_figures, selected_table, validate_pool,
)

# Widgetsız kullanımda da bütün ayarlar buradan yönetilir.
AYARLAR = Settings()  # Hızlı canlı demo, 50 test / veri 83 / algoritma 42
# Tam plan için AYARLAR = Settings.full()
deney = PresentationExperiment(AYARLAR, root=ROOT)
try:
    import ipywidgets
    from kod.raporlama.sunum_paneli import PresentationPanel
    panel = PresentationPanel(deney)
except ImportError:
    panel = None
    print('Widget bulunamadı. Aşağıdaki widgetsız hücrelerle aynı hesaplama yapılabilir.')

## Ayar paneli
Hazır mod, veri ve arama ayarları burada değişir. Algoritma tohumlarının sayısı tekrar sayısıdır. Tam mod 315 koşumdur. Greedy her veri/boyutta bir kez çalışır.

**Veriyi değiştirdikten sonra “Veri değişikliğini uygula”, ardından “Deneyi çalıştır / Sonuçları güncelle” düğmesine basın.** Grafik ve koşum menüleri yalnız görünümü değiştirir.

İleri ayarlar mevcut IBMGO seçeneklerini JSON olarak alır. Örnek `{"spiral_search_trials": 2}`. Bütçe bu adayları da içerir. Boyuta duyarlı politika varsayılan olarak açıktır. Yerel deneme sayısı panelden ayrıca belirlenir.

In [2]:
if panel is not None:
    display(panel.settings_view)
else:
    display(pd.Series(vars(AYARLAR), name='Ayarlar').to_frame())
    AYARLAR.validate()
    display(pd.DataFrame([dict(boyut=n, **AYARLAR.budget(n)) for n in AYARLAR.sizes]))

## 1 · Problem ve telekom örneği
**12 ay %20 indirim** kampanyasında ilk aktivasyon bir kayıt oluşturur. Aynı istek kimliği yeniden gönderilince ikinci bir kayıt oluşmamalıdır. İlk fatura indirimi de doğru uygulanmalıdır.

Fatura ve tekrar istek testlerinin ön koşulu aktivasyon testidir. Yalnız süreyi azaltmak bu ilişkileri bozabilir.

In [3]:
ornek = deney.pool(50, 83)
telekom = pd.DataFrame(ornek['metadata']['test_cases'])
display(telekom.iloc[[0, 4, 7]][[
    'id', 'offer_name', 'title', 'request_id', 'expected_result',
    'expected_monthly_amount_tl', 'requirement_ids', 'prerequisite_test_ids',
]])

,id,offer_name,title,request_id,expected_result,expected_monthly_amount_tl,requirement_ids,prerequisite_test_ids
0,T_0001,12 Ay %20 Indirim,12 Ay %20 Indirim - Uygun musteri icin ilk akt...,REQ_KMP01_000_activation_success,AKTIF / tek yeni kayit,NaN,"(G01, G03, G05, G13, KMP01_E, KMP01_A)",()
4,T_0005,12 Ay %20 Indirim,12 Ay %20 Indirim - Aktif kampanyanin ilk fatu...,REQ_KMP01_000_first_bill,BASARILI / beklenen ucret ve kota,160.0,"(G07, KMP01_B, KMP01_A)","(T_0001,)"
7,T_0008,12 Ay %20 Indirim,12 Ay %20 Indirim - Ayni istek kimligiyle ikin...,REQ_KMP01_000_activation_success,BASARILI / sadece bir kayit,NaN,"(G10, KMP01_A)","(T_0001,)"


## 2 · Veri ve sunum kopyası
Her testin kapsam hedefleri, sentetik süresi ve ön koşulları var. Genişletilmiş veri üreticisi kullanılır. Veri tohumu bağlam ve süreleri belirler.

Aşağıdaki görünüm en fazla 8×8 kapsam hücresi gösterir. Test ve hedef indeksleri sıfırdan başlar. ★ kritik hedefi gösterir. Süreyi, kritiklik işaretini ve tek bir kapsam hücresini değiştirip uygulayın. Kaynak dosyaya yazılmaz. **Orijinal veriye dön** yalnız seçili havuzun değişikliklerini kaldırır. Bütün yöntemler aynı kopyayı alır.

In [4]:
if panel is not None:
    display(panel.editor)
else:
    display(pd.DataFrame(ornek['coverage'][:8, :8]))
    # Widgetsız veri değişikliği örneği. Uygulamak için yorum işaretlerini kaldırın.
    # kopya = deney.pool(50, 83)
    # kopya['durations'][4] = 75.0
    # kopya['coverage'][4, 6] = 0
    # kopya['critical'] = np.array([1, 3, 4, 5, 6], dtype=int)
    # deney.set_pool(50, 83, kopya)
    # deney.reset_pool(50, 83)  # Orijinal veriye dön

## 3 · Karar değişkeni
Her test için $x_i \in \{0,1\}$. 1 seçildi, 0 seçilmedi demektir.

Örneğin $x_{fatura}=1$ ise aktivasyon da seçilmelidir. Paket önce seçilir, sonra ön koşulları koruyan bir sıraya konur.

## 4 · Zorunlu kısıtlar
- Kritik gereksinim kapsamı %100.
- Genel gereksinim kapsamı en az %90. Panelden değiştirilebilir.
- Ön koşul ihlali sıfır. Ebeveyn test indeksi çocuk testten küçük olmalıdır.

Bütün havuzlar arama başlamadan kontrol edilir. Kapsanamayan kritik hedef, olanaksız genel kapsam veya sıralı olmayan bağımlılık grafiği hata mesajıyla bildirilir.

## 5 · Amaç fonksiyonu
**Düşük fitness daha iyi.** Uygun paketlerde

$$F=\frac{w_t(T_s/T)+w_n(N_s/N)+w_r R+w_q Q}{w_t+w_n+w_r+w_q}.$$

$R$ tekrarlı kapsam isabetlerinin toplam isabete oranıdır. $Q$, kapsanmayan kritik olmayan hedeflerin ters kapsama sıklığıyla ağırlıklandırılan kaybıdır.

Ana ayarlar süre 0,40, test sayısı 0,40, tekrar 0,20, nadir kayıp 0. Ağırlıklar toplamlarına bölünerek normalize edilir. Toplamın 1 olması gerekmez. Negatif, sonsuz veya toplamı sıfır olan ağırlıklar reddedilir. Kısıt ihlalinde 1000 + ihlal başına 100 ceza ve amaç puanı uygulanır. Farklı büyüklüklerin fitness değerlerini doğrudan başarı sıralaması yapmayın.

## 6 · MGO → BMGO → IBMGO
**MGO** sürekli uzayda dört hareketle aday üretir. **BMGO** konum farkını V-dönüşümüyle bit çevirme olasılığına dönüştürür. Ardından kapsam ve ön koşullar onarılır, gereksiz testler budanır.

**Bu projedeki IBMGO**, çeşitli başlangıç ve yerel aramayı boyuta duyarlı hareket seçenekleriyle birleştiren probleme özgü uyarlamadır. Özgün MGO veya bir makalenin birebir uygulaması olarak sunulmaz.

Boyut ≤250 için adaptif bit dönüşümü ve tekli takas, 251–750 için referans hareketleri / ICMIC / doğrusal olmayan kontrol ve çoklu komşuluk, 751+ için %50 hareket karışımı ve tekli komşuluk kullanılır. Politika ve veri üreticisi bu düzenlemede değiştirilmedi.

## 7 · Deney ayarları ve bütçe
Ana arşiv 5 boyut × 3 veri tohumu × (10 IBMGO + 10 GA + 1 Greedy) = **315 koşum** içerir. IBMGO ve GA tohumları 42–51.

Standart plan IBMGO için $12+20(4×12+12)=1212$, GA için $12+100×12=1212$ çağrıdır. Popülasyon, iterasyon ve ek aday sayıları değişince GA nesli otomatik hesaplanır. Tam eşitlik mümkün değilse en yakın uygulanabilir bütçe ve GA−IBMGO farkı panelde görünür.

Planlanan bütçe ve gerçek çağrı sayısı ayrı kaydedilir. Yerel arama uygun aday bulamadığında gerçek sayı azalabilir. Aynı paketi tekrar puanlamak benzersiz aday değildir. Greedy eşit bütçeli bir popülasyon yöntemi değildir. Eşit çağrı sayısı eşit arama süresi değildir.

## 8 · Sonuç kaynağını seç ve deneyi çalıştır
Aşağıdaki hücre yalnız **Kayıtlı deney** dosyalarını okur. Canlı panel varsayılan olarak hızlı demo ayarındadır. Arşiv ayarları sonuç yanında ayrıca görünür.

Yeni hesaplama yalnız düğmeyle başlar. Ayar veya uygulanan veri değişikliği sonuçları **Güncel değil** yapar. Hesaplama tüm yöntemleri aynı girdilerle çalıştırır, sonuçları birlikte yeniler. Filtre değiştirmek optimizasyon başlatmaz.

Her yeni çalışma `sonuclar/sunum_calistirmalari/<benzersiz_kimlik>` içine ayarları, girdi kopyalarını, kod özetlerini, ham/özet sonuçları ve izleri yazar. Başarısız çalışma da durum dosyası bırakır. Eski dosyalar korunur.

In [5]:
if panel is not None:
    panel.archive()
else:
    deney.load_archive()
    print(deney.result['source'], deney.result['note'])

## 9 · Sonuç tabloları ve grafikler
Bu ekran yöntemlerin fitness, süre tasarrufu, tahmini paket süreleri, seçilen test, arama süresi ve kapsamını aynı sonuç tablosundan çizer. Ham kayıtlar ayrı açılır.

Ortalama grafiğinin hata çubukları aynı boyuttaki veri ve algoritma tekrarlarının **örnek standart sapmasıdır**. Greedy ana arşivde yalnız 3 veri kaydına sahiptir. Arşivde bulunmayan süre toplamları boş gösterilir. Sabit PNG kullanılmaz.

In [6]:
if panel is not None:
    display(panel.result_view)
else:
    print(deney.result['source'])
    display(pd.Series(deney.result['settings']).to_frame('Sonuç ayarları'))
    display(summarize(deney.result['rows']))
    fig = result_figures(deney.result, 500)
    display(fig)
    plt.close(fig)

## 10 · İterasyon ve yakınsama
Yeni hesaplamadan sonra **Koşum** menüsünden boyut, veri ve algoritma tohumunu seçin.

IBMGO grafiği iterasyonu, GA grafiği nesli gösterir. Sıfırıncı nokta başlangıç popülasyonunun en iyi puanıdır. Ortak grafik gerçek kümülatif amaç fonksiyonu çağrısı üzerindedir. Yerel ve spiral aramadaki iyileşmeler bu eksende yer alır.

Seçili veri ve boyuttaki diğer tekrarlar silik çizilir. Farklı veri kümeleri bu eğrilerde birleştirilmez. Özet tabloda başlangıç, nihai fitness, çağrı sayısı, adım sayısı, nihai en iyiye ilk erişim, son iyileşme ve arama süresi yer alır.

“Nihai en iyiye ilk erişim” o koşumun sonuna kadar bulunan en iyi değere ilk erişimdir. Tolerans `rtol=1e-10, atol=1e-12`. Bu değer küresel optimum veya kesin yakınsama kanıtı değildir. Eski CSV’lerde iz bulunmadığı için yakınsama grafiği yeni çalıştırma gerektirir.

## 11 · Seçilen testler ve sıralama
Yeni sonuç ekranında seçili koşumun ilk 20 testi, süreleri ve ön koşulları sıralı gösterilir. Tüm sıra yeni çalışma klasöründeki `izler_ve_secimler.json` içindedir.

Önceliklendirme, ön koşulu tamamlanan testler arasında saniye başına yeni kritik kapsamı tercih eder. Hata tespit matrisi olmadığı için APFD ve gerçek hata keşif hızı hesaplanmaz.

## 12 · Bulgular ve sınırlar
Ana arşivde 50 testte IBMGO ve GA eşit, 250 ve 500 testte IBMGO daha çok eşleşmede daha düşük fitness veriyor. 1000 testte GA daha çok eşleşmede üstün. Sayısal karşılaştırmayı üstteki gerçek kayıtlardan okuyun.

Veri ve süreler sentetiktir. Üç veri tohumu gerçek dünya genellemesi için sınırlıdır. Tam ilk-BMGO ve bileşen ablasyonu bu pakette yok. Seed 71 doğrulaması ve `nihai_bmgo` sonuçları mevcut pakette bulunmuyor. Bunlara dayanarak doğrulanmış başarı iddiası kurulmaz.

Canlı demo, ana arşiv ve ek ölçek deneyi ayrı kaynaklardır. Canlı demo tek koşumu dayanıklılık ortalaması gibi sunulmaz.

## Widgetsız yeniden hesaplama
Aşağıdaki hücre varsayılan olarak hesaplama başlatmaz. Widget kullanılamıyorsa `CALISTIR = True` yapıp hücreyi çalıştırın. Ayarları üstteki `AYARLAR` nesnesinde değiştirin. Aynı deney motoru, veri doğrulaması ve kayıt biçimi kullanılır.

In [7]:
CALISTIR = False
if CALISTIR:
    deney.settings = AYARLAR
    yeni = deney.run(lambda tamam, toplam, ad: print(f'{tamam}/{toplam} · {ad}'))
    print(yeni['source'], yeni['paths'])
    display(summarize(yeni['rows']))
    fig = result_figures(yeni, AYARLAR.sizes[0])
    display(fig)
    plt.close(fig)
    anahtar = next(k for k, r in yeni['runs'].items() if r['method'] == 'IBMGO')
    fig = convergence_figures(yeni, anahtar)
    display(fig)
    plt.close(fig)
    display(selected_table(yeni, anahtar))